In [5]:
!unzip "/content/drive/MyDrive/MRI-Images-of-Brain-Tumor.zip" -d "/content/"

Streaming output truncated to the last 5000 lines.
  inflating: /content/MRI-Images-of-Brain-Tumor/test/no-tumor/Tr-no_1574_jpg.rf.f3e0cd34f109acab9c8f366ca8967a76.jpg  
  inflating: /content/MRI-Images-of-Brain-Tumor/test/no-tumor/Tr-no_1575_jpg.rf.2b4235401d0914690c199fbd9b2f50f5.jpg  
  inflating: /content/MRI-Images-of-Brain-Tumor/test/no-tumor/Tr-no_1584_jpg.rf.3a22fa5ee37e7011c0c75e8e3871e927.jpg  
   creating: /content/MRI-Images-of-Brain-Tumor/test/pituitary/
  inflating: /content/MRI-Images-of-Brain-Tumor/test/pituitary/Tr-piTr_0001_jpg.rf.f7ceded844befc4fe44a7d07bf2a2fc0.jpg  
  inflating: /content/MRI-Images-of-Brain-Tumor/test/pituitary/Tr-piTr_0009_jpg.rf.a2e16deedde817260e0555b73f44e94c.jpg  
  inflating: /content/MRI-Images-of-Brain-Tumor/test/pituitary/Tr-pi_0024_jpg.rf.269f5e4416ebb23a6c350bc246bff746.jpg  
  inflating: /content/MRI-Images-of-Brain-Tumor/test/pituitary/Tr-pi_0033_jpg.rf.267680b1481bd5bd901f3260f6fbe7db.jpg  
  inflating: /content/MRI-Images-of-Brain-Tu

In [6]:
import os
os.listdir('/content/MRI-Images-of-Brain-Tumor')

['train', 'valid', 'test']

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [10]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [11]:
train_data = datasets.ImageFolder(
    "/content/MRI-Images-of-Brain-Tumor/train",
    transform=transform
)

valid_data = datasets.ImageFolder(
    "/content/MRI-Images-of-Brain-Tumor/valid",
    transform=transform
)

test_data = datasets.ImageFolder(
    "/content/MRI-Images-of-Brain-Tumor/test",
    transform=transform
)

In [12]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

In [13]:
print(train_data.classes)

['glioma', 'meningioma', 'no-tumor', 'pituitary']


In [14]:
class BrainTumorCNN(nn.Module):

    def __init__(self):
        super(BrainTumorCNN, self).__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(3,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*28*28,512),
            nn.ReLU(),
            nn.Linear(512,4)
        )

    def forward(self,x):
        x = self.conv(x)
        x = self.fc(x)
        return x

In [15]:
model = BrainTumorCNN().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [16]:
epochs = 10

for epoch in range(epochs):

    model.train()
    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

Epoch 1, Loss: 0.5091219257752774
Epoch 2, Loss: 0.19595157511342884
Epoch 3, Loss: 0.10753365988858928
Epoch 4, Loss: 0.05812616561938046
Epoch 5, Loss: 0.06495507006578827
Epoch 6, Loss: 0.03105222947463409
Epoch 7, Loss: 0.05282332445988116
Epoch 8, Loss: 0.011197565209316716
Epoch 9, Loss: 0.0030310862805243356
Epoch 10, Loss: 0.0031268038540597986


In [17]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in valid_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Validation Accuracy:", 100 * correct / total)

Validation Accuracy: 98.22926374650513


In [18]:
correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Test Accuracy:", 100 * correct / total)

Test Accuracy: 99.25512104283054


In [23]:
torch.save(model.state_dict(), "brain_tumor_model.pth")